# News Articles Grouping Research

The central hypothesis of this research is that by combining multiple layers of processing and analysis—rather than relying solely on semantic or topical similarity—we can more accurately group news articles into meaningful real-world events.

In other words, we do not want to cluster articles together simply because they mention the same high-level entity (e.g., the President of Argentina). Instead, we aim to group them based on the specific event that occurred. For example:

- *The Argentine president approves a set of laws preventing X, Y, and Z.*
- *The Argentine president vetoes a tax reform proposal.*

Although both articles mention the same public figure, they describe different real-world events and therefore should belong to different clusters.

---

## Baseline

As a baseline, we preprocess the articles by generating embeddings using `multilingual-e5-large`. This model was selected because it has been trained on diverse multilingual data, including news-style corpora, making it well-suited for capturing the semantic structure of journalistic text. Additionally, it supports both English and Spanish,the two languages evaluated in this experiment, allowing us to maintain a unified embedding space across languages.

In cases where an article exceeds the model's maximum context length, we truncate it rather than average multiple chunks. Averaging chunk embeddings may dilute event-specific signals and introduce smoothing effects that blur important distinctions. By applying a head-based truncation strategy with 512 tokens, we aim to preserve the most informative portion of the article, as news writing typically presents the core event details in the opening paragraphs.

Future experiments will explore alternative embedding models to assess their impact on clustering performance. However, `multilingual-e5-large` provides a strong and appropriate baseline for this task.

After embedding, we apply **HDBSCAN**, a density-based clustering algorithm that:

- Identifies clusters of varying density.
- Does not require specifying the number of clusters in advance.
- Labels outliers as noise.

<br/>

| Metric | Score |
|------|------|
| Homogeneity Score | 0.9898 |
| Completeness Score | 0.7510 |
| V-Measure Score | 0.8540 |
| Intra-cluster Similarity | 0.8821 |

<br/>

HDBSCAN groups articles based on proximity in the embedding space. While it performs very well for clearly separated events, in many cases it clusters articles according to broad semantic similarity or shared topics rather than the specific real-world event being described. Additionally, the baseline model labeled **~38.7% of the articles as noise** in order to produce safer clusters.

This limitation highlights the challenges of relying solely on **embedding-based semantic clustering** for event-level grouping. Nevertheless, this experiment provides a solid **baseline and reference point**, allowing us to analyze the limitations of purely semantic clustering and guide improvements to the pipeline in subsequent stages.

---

## Research Structure

### Stage 1 — Entity Extraction & Disambiguation

In this stage, we extract named entities from the articles and resolve them to unique identifiers in a knowledge base. The process consists of three main steps:

- **Sentence Splitting**  
  Each article is first divided into individual sentences, as the subsequent processing steps operate at the sentence level rather than on full documents. For this step, we use the **spaCy sentence splitter** with the `en_core_web_sm` model.

- **Named Entity Recognition (NER)**  
  For each sentence, we extract all detected entities and retain them based on a predefined confidence threshold. This step is implemented using the **Transformers library** with the `Jean-Baptiste/roberta-large-ner-english` model.

- **Entity Disambiguation**  
  Extracted entities are then mapped to **Wikipedia titles identifiers**, allowing entities to be represented in a standardized and unambiguous form. To facilitate this process, entities are marked using the format:  
  `"... [START] {entity} [END] ..."`  
  This helps the disambiguation model identify the target span within the text. The disambiguation process is implemented using the **Transformers library** together with the `facebook/mgenre-wiki` model.

  Additional experimentation will be conducted to handle cases where an entity cannot be confidently matched to the knowledge base.

<br/>

### Stage 2 — Article Pair Generation

This stage focuses on identifying candidate article pairs that are likely to describe similar events.

- **Article Preprocessing**  
  All articles are embedded using the `multilingual-e5-large` model, producing a **1024-dimensional dense vector representation** for each article.

- **Similarity Search Algorithms**  
  We evaluate two approaches for retrieving similar articles:

  - **k-Nearest Neighbors (KNN)**  
    Performs exact similarity search, prioritizing retrieval accuracy at the cost of computational efficiency.

  - **Approximate Nearest Neighbors (ANN)**  
    Sacrifices a small amount of retrieval accuracy in exchange for significantly improved speed and scalability, returning results that are sufficiently close in embedding space.

  Both methods are evaluated to determine the best trade-off between **retrieval quality and computational performance**.

- **Pair Construction**  
  For each article, we retrieve its top-*k* most similar neighbors (e.g., 100). Candidate pairs are then constructed as:

  `(A, A₁), (A, A₂), ..., (A, Aₖ)`

  These candidate pairs are passed to the next stage for deeper event-level comparison.

<br/>

### Stage 3 — Pair Comparison

The objective of this stage is to determine whether two articles within a candidate pair describe the **same real-world event**. Two complementary methods are used:

- **Cross-Encoder Classification Model**  
  A custom fine-tuned transformer model takes both articles jointly as input and outputs a probability indicating whether they refer to the same event.  
  Unlike bi-encoder embedding approaches, a **cross-encoder directly models interactions between both texts**, enabling more precise semantic comparison.

- **Entity Similarity Analysis**  
  We compute the **Jaccard similarity** between the sets of **normalized entities** extracted and disambiguated in Stage 1. This produces an **entity overlap score** that reflects how many entities are shared between the two articles.

Each method produces a similarity score. These scores are then **combined using a weighted aggregation** to compute a final **event-level similarity score** for each article pair.

<br/>

### Stage 4 — Graph Generation

After computing similarity scores for all candidate pairs, we construct an **article similarity graph**:

- Each **node** represents an article.
- An **edge** between two nodes represents their final similarity score.
- The graph is **sparse**, since each article is connected only to its top-*k* nearest neighbors.
- Weak edges (below a predefined threshold) are removed to reduce noise.

Once the graph is constructed, we apply **Leiden Community Detection**, a graph-based clustering algorithm that identifies communities by optimizing modularity while ensuring well-connected and stable clusters.

By comparing these methods, we aim to determine which approach produces the most **coherent event-level clusters**. The best-performing configuration will be used in the final research pipeline.

---

## 0. Notebook Setup

#### Import Dependencies

In [ ]:
!pip install cugraph-cu12 --extra-index-url=https://pypi.nvidia.com

In [ ]:
!pip install faiss-gpu-cu12

In [3]:
import os
import faiss
import numpy as np
import pandas as pd
import torch
import gc
import re
import ast
import spacy
import cugraph

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline, AutoModelForTokenClassification, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from sklearn.metrics import homogeneity_completeness_v_measure
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
# Setup Google Drive in Colab
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#### Load Dataset

All articles should have values based on the data analysis notebook results but just in case we add the extra check.

In [6]:
DATASET_PATH = "/content/drive/MyDrive/Research/articles_grouping/datasets/preprocessed/main/articles_events_2500.csv"

In [23]:
df = pd.read_csv(filepath_or_buffer=DATASET_PATH)
df = df.reset_index(drop=True)

df["title"] = df["title"].fillna("")
df["content"] = df["content"].fillna("")
df["text"] = "passage: " + df["title"] + "\n" + df["content"] # How the embedder needs to receive the corpus

print(f"Dataset loaded and formatted successfully with {df.shape[0]} examples")

Dataset loaded and formatted successfully with 163753 examples


---

## 1. Entity Extraction & Disambiguation

We will setup all 3 entity steps and finally run the whole pipeline.

> *Note: Results will be save into a `.csv` in order to avoid re-calculating this pipeline on every run of the research notebook.*

### 1.1 SpaCy Sentence Splitter

#### Download spacy Model

In [ ]:
!python -m spacy download en_core_web_sm

#### Load spaCy Model

In [ ]:
spacy_model = spacy.blank("en")
spacy_model.add_pipe("sentencizer")

In [ ]:
def sentence_splitter(batch, batch_size = 128):
    sentences = []

    texts = batch["content"]
    for doc in spacy_model.pipe(texts, batch_size=batch_size):
        sents = [sent.text for sent in doc.sents]
        sentences.append(sents)

    batch["sentences"] = sentences
    return batch

### 1.2 Name Entity Recognition

#### Load Model & Tokenizer

In [ ]:
NER_MODEL_NAME = "Jean-Baptiste/roberta-large-ner-english"

ner_tokenizer = AutoTokenizer.from_pretrained(NER_MODEL_NAME)
ner_model = AutoModelForTokenClassification.from_pretrained(
    NER_MODEL_NAME,
    torch_dtype=torch.float16).eval()

#### Setup Model Pipeline

In [ ]:
ner_pipeline = pipeline("ner",
                        model=ner_model,
                        tokenizer=ner_tokenizer,
                        aggregation_strategy="simple",
                        device="cuda")

In [ ]:
def normalize_entity(entity):
    if entity.isupper() and len(entity.split()) == 1:
        return entity
    return entity.strip().title()

In [ ]:
def name_entity_recognition(batch, batch_size = 128):
    all_sentences = [sent for article_sents in batch["sentences"] for sent in article_sents if sent.strip()]

    all_entities_raw = ner_pipeline(all_sentences, batch_size=batch_size)
    all_entities = squared_list = [[{
        "entity": normalize_entity(ent.get("word").strip()),
        "type": ent.get("entity_group").strip().upper()
    } for ent in sentence_entities if ent.get("word").strip() != "."] for sentence_entities in all_entities_raw]

    entities = []
    idx = 0
    for article_sents in batch["sentences"]:
        n = len(article_sents)
        entities.append(all_entities[idx:idx+n])
        idx += n

    batch["entities"] = entities
    return batch

### 1.3 Build Entity Sentences

The entity disambiguation model requires each target entity to be wrapped with special markers `[START]` and `[END]`, along with the full sentence in which the entity appears. This provides the surrounding **context**, helping the model better determine which real-world entity is being referenced.

For each detected entity, we construct a sentence in the following format:

`"... [START] {entity} [END] ..."`

Providing the sentence context allows the model to leverage **semantic cues from the surrounding text**, improving the accuracy of the entity disambiguation process.

In [ ]:
def build_tagged_sentences(batch):
    all_tagged = []

    for article_sents, article_ents in zip(batch["sentences"], batch["entities"]):
        article_tagged = []

        for sent, sent_ents in zip(article_sents, article_ents):
            for ent in sent_ents:
                tagged = sent.replace(ent["entity"], f" [START] {ent['entity']} [END] ", 1)

                article_tagged.append({
                    "tagged": tagged,
                    "entity": ent["entity"].strip(),
                    "type": ent["type"],
                })
        all_tagged.append(article_tagged)

    batch["tagged_sentences"] = all_tagged
    return batch

### 1.4 Entity Disambiguation

In this section, we use the tagged sentences generated in **Step 1.3** together with the `facebook/mgenre-wiki` transformer model to perform **entity disambiguation**.

For each tagged entity, the model receives the full sentence with the entity wrapped in `[START]` and `[END]`. Using the surrounding context, the model predicts the **top 5 most probable Wikipedia titles** corresponding to that entity mention.

We select the candidate with the **highest probability** and normalize the resulting title so it can be used as a **unique identifier for the entity** within the pipeline.

To improve performance, we also implement a **lookup table (cache)** for previously processed entities. If the system encounters an entity with the **same surface form and entity type** that has already been disambiguated, the normalized identifier can be retrieved directly from the lookup table instead of running the disambiguation model again.

This caching mechanism significantly reduces redundant computations and improves the efficiency of the entity normalization process.

#### Load Model & Tokenizer

In [ ]:
MODEL_NAME = "facebook/mgenre-wiki"

In [ ]:
disambiguation_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
disambiguation_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device).eval()

#### Prepare Step Functions

In [ ]:
ABBREVIATION_RE = re.compile(r'^[A-Z\.]{1,5}$|^([A-Z]\.\s*)+$')

def is_abbreviation(s):
        return bool(ABBREVIATION_RE.match(s.strip()))

def candidate_rank(candidate):
        id_part = candidate.split(" >> ")[0].strip()

        if not is_abbreviation(id_part) and len(id_part.split()) <= 3:
            return (0, len(id_part))

        if not is_abbreviation(id_part):
            return (1, len(id_part))

        return (2, len(id_part))

def get_id(candidates, entity_type):
    if not candidates:
        return None

    return min(candidates, key=candidate_rank).split(" >> ")[0]

In [ ]:
lookup_table = {}

def generate_key(entity, entity_type):
    return f"{entity.upper().replace(" ", "_")}_{entity_type.upper()}"

def disambiguate_batch(batch):
    misses = []
    # Build lookup table
    for article_tagged in batch["tagged_sentences"]:
        for ent in article_tagged:
            key = generate_key(entity=ent["entity"],
                               entity_type=ent["type"])

            if key not in lookup_table:
                lookup_table[key] = None
                misses.append((key, ent["tagged"], ent["type"]))

    if len(misses) > 0:
        MGENRE_BATCH_SIZE = 64
        keys, tagged_sentences, entity_types = zip(*misses)

        all_ids = []
        for i in range(0, len(tagged_sentences), MGENRE_BATCH_SIZE):
            sub_sentences = list(tagged_sentences[i:i+MGENRE_BATCH_SIZE])

            inputs = disambiguation_tokenizer(
                sub_sentences,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512
            ).to(device)

            outputs = disambiguation_model.generate(
                **inputs,
                num_beams=5,
                num_return_sequences=5,
            )

            decoded = disambiguation_tokenizer.batch_decode(outputs, skip_special_tokens=True)
            chunked = [decoded[j*5:(j+1)*5] for j in range(len(sub_sentences))]
            all_ids.extend(chunked)

        for key, candidates, entity_type in zip(keys, all_ids, entity_types):
            lookup_table[key] = get_id(candidates, entity_type)

    # Reconstruct output using lookup table
    all_raw_entities = []
    all_canonical_ids = []

    for article_tagged in batch["tagged_sentences"]:
        article_raw = []
        article_ids = []
        for ent in article_tagged:
            key = key = generate_key(entity=ent["entity"], entity_type=ent["type"])
            article_raw.append({"entity": ent["entity"], "type": ent["type"]})
            article_ids.append(lookup_table[key])
        all_raw_entities.append(article_raw)
        all_canonical_ids.append(article_ids)

    batch["raw_entities"] = all_raw_entities
    batch["canonical_ids"] = all_canonical_ids
    return batch

### 1.5 Titles Normalization

In [ ]:
def replace_parens_with_underscore(match):
    content_in_parens = match.group(1).strip()
    if content_in_parens:
        return f"_{content_in_parens.upper()}"
    return ""

def clean_and_normalize_label(label):
    if label is None:
        return None

    original_label = str(label)

    processed_label = re.sub(r'\s*\(([^)]*)\)', replace_parens_with_underscore, original_label)

    normalized_label = processed_label.upper()
    normalized_label = re.sub(r'[^A-Z0-9_]+', '_', normalized_label)
    normalized_label = re.sub(r'_+', '_', normalized_label)
    normalized_label = normalized_label.strip('_')

    return normalized_label

def normalize_and_deduplicate_canonical_ids(batch):
    all_normalized_ids = []

    for article_canonical_ids in batch["canonical_ids"]:
        normalized_unique_ids = set()

        for entity_id in article_canonical_ids:
            if entity_id:
                normalized_id = clean_and_normalize_label(entity_id)
                if normalized_id:
                    normalized_unique_ids.add(normalized_id)
        all_normalized_ids.append(list(normalized_unique_ids))

    batch["normalized_canonical_ids"] = all_normalized_ids
    return batch

### 1.6 Run Entity Pipeline

In this final subsection, we execute the complete **entity processing pipeline**, combining all previously defined steps.

To improve efficiency, the pipeline processes the data in **batches**. The input to the pipeline is a **DataFrame containing the information for each article**, and the output is an **augmented copy of that DataFrame** that includes the extracted entities and their corresponding **normalized identifiers**.

These identifiers represent the **disambiguated entities** obtained during the entity linking stage and will later be used for **entity similarity calculations** in subsequent stages of the pipeline.

In [ ]:
ds = Dataset.from_pandas(df)

if not os.path.exists("checkpoint_1"):
    ds = ds.map(sentence_splitter, batched=True, batch_size=128)
    ds.save_to_disk("checkpoint_1")
else:
    ds = Dataset.load_from_disk("checkpoint_1")

if not os.path.exists("checkpoint_2"):
    ds = ds.map(name_entity_recognition, batched=True, batch_size=512)
    ds.save_to_disk("checkpoint_2")
else:
    ds = Dataset.load_from_disk("checkpoint_2")

if not os.path.exists("checkpoint_3"):
    ds = ds.map(build_tagged_sentences, batched=True, batch_size=256)
    ds.save_to_disk("checkpoint_3")
else:
    ds = Dataset.load_from_disk("checkpoint_3")

if not os.path.exists("checkpoint_4"):
    ds = ds.map(disambiguate_batch, batched=True, batch_size=64)
    ds.save_to_disk("checkpoint_4")
else:
    ds = Dataset.load_from_disk("checkpoint_4")

if not os.path.exists("checkpoint_5"):
    ds = ds.map(normalize_and_deduplicate_canonical_ids, batched=True, batch_size=32)
    ds.save_to_disk("checkpoint_5")
else:
    ds = Dataset.load_from_disk("checkpoint_5")

# Final save
ds.to_pandas().to_csv("entity_results.csv", index=False)

### Pipeline Execution Time

| Step | Time |
|-----|-----|
| Step 1 | 10 minutes |
| Step 2 | 1 hour 58 minutes |
| Step 3 | 3 minutes 56 seconds |
| Step 4 | 1 hour 18 minutes |
| Step 5 | 42 seconds |
| **Total Time** | **3 hours 30 minutes 38 seconds** |

### 1.7 Load DataFrame with Entities

This section acts as a **checkpoint** in the pipeline. Instead of recomputing the entire entity extraction and disambiguation process, we load a previously saved **DataFrame containing the extracted entities and their normalized identifiers**.


In [7]:
DATASET_ENTITIES_PATH = "/content/drive/MyDrive/Research/articles_grouping/datasets/results/intermediate/entity_results_clean.csv"

In [17]:
df_entities = pd.read_csv(filepath_or_buffer=DATASET_ENTITIES_PATH)
df_entities = df_entities.reset_index(drop=True)

print(f"Loaded entities dataset containing {len(df_entities)} examples")

Loaded entities dataset containing 163753 examples


---

## 2. Article Pair Generation

### 2.1 Data Preparation

In this section, we generate embeddings for approximately ~160k news articles using a combination of each article's **title and content** as input. This approach provides a richer contextual representation than using either component independently.

For embedding generation, we use the `sentence_transformers` library within a Google Colab environment. If the experiment is replicated locally, the same procedure can be executed using `Ollama`, provided the model is properly installed and configured.

As introduced earlier, we use `multilingual-e5-large`, a multilingual embedding model that supports both English and Spanish and is trained to produce high-quality semantic representations across languages. Its training data includes web and news-style corpora, making it particularly suitable for journalistic text.

To ensure efficient memory management, we implement a batching strategy during embedding generation. The computed embeddings are stored on disk, allowing us to generate them once and reuse them across multiple experiments without recomputation. This guarantees both computational efficiency and experimental consistency.

#### Load Embedder Model

In [ ]:
MODEL_NAME = "intfloat/multilingual-e5-large"

In [ ]:
emb_model = SentenceTransformer(MODEL_NAME, trust_remote_code=True)

In [24]:
TOTAL = len(df)
EMBEDDING_DIM = 1024
BATCH_SIZE = 2048

In [ ]:
EMBEDDINGS_OUTPUT_PATH = "/content/drive/MyDrive/Research/articles_grouping/embeddings_2500.dat"

In [ ]:
embeddings_mmap = np.memmap(EMBEDDINGS_OUTPUT_PATH, dtype="float32", mode="w+", shape=(TOTAL, EMBEDDING_DIM))

for i in tqdm(range(0, TOTAL, BATCH_SIZE)):
    batch = df["text"].iloc[i : i + BATCH_SIZE].tolist()
    embeddings_mmap[i : i + BATCH_SIZE] = emb_model.encode(
        batch,
        batch_size=256,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    if i % (BATCH_SIZE * 10) == 0:
        embeddings_mmap.flush()

del embeddings_mmap

100%|██████████| 80/80 [44:05<00:00, 33.07s/it]


To efficiently utilize the available computational resources during embedding generation, we implemented a two-level batching strategy:

- **Data loading batch size:** 2,048 articles per batch.  
  This prevents loading the entire dataset into memory at once, reducing RAM usage and ensuring stability during processing.

- **Model inference batch size:** 256 articles per forward pass.  
  This value was selected to maximize GPU utilization while avoiding out-of-memory (OOM) errors.

This stage of the experiment was executed in a Google Colab Pro environment using an **NVIDIA A100 GPU with 40GB of VRAM**.  

Processing the full dataset of **163,753  articles** required **44 minutes, and 5 seconds**, including batching and disk storage of embeddings.

#### Load Embeddings

The embeddings were precomputed and stored on disk. If the embedding generation step has already been executed in the current session, the following cell can be skipped.

In [25]:
DRIVE_PATH = "/content/drive/MyDrive/Research/articles_grouping/datasets/preprocessed/main/embeddings_2500.dat"

In [26]:
embeddings = np.fromfile(DRIVE_PATH, dtype="float32").reshape(TOTAL, EMBEDDING_DIM)

In [27]:
print(f"Embeddings loaded with shape: {embeddings.shape}")
print(f"Total number of articles (rows): {embeddings.shape[0]}")
print(f"Embedding dimension (columns): {embeddings.shape[1]}")

Embeddings loaded with shape: (163753, 1024)
Total number of articles (rows): 163753
Embedding dimension (columns): 1024


### 2.2 Similarity Search Algorithm

In this section, we evaluate and compare **exact k-Nearest Neighbors (KNN)** and **Approximate Nearest Neighbors (ANN)** based on both accuracy and computational efficiency. The goal is to determine which method is better suited for large-scale similarity search over ~160k news article embeddings.

Given the size of the dataset, exact KNN can become computationally expensive, especially when computing similarities across the full embedding space. While it provides precise nearest neighbors, its scalability may be limited as the dataset grows.

ANN methods, on the other hand, trade a small amount of retrieval accuracy for significantly improved speed and scalability. These methods aim to retrieve neighbors that are sufficiently close to the true nearest neighbors, making them practical for large embedding collections.

We compare both approaches in terms of:

- **Retrieval quality** (e.g., recall@k or overlap with exact neighbors)
- **Query latency and indexing time**
- **Scalability with increasing dataset size**

Based on this evaluation, we will select the most appropriate similarity search strategy for the remainder of the research pipeline.

> *Note: For the implementation we will use Meta's library called **FAISS**, which has built-in implementations for both algorithms.*

In [28]:
K_NEIGHBORS = 100

### K-Nearest Neighbors (KNN)

In [29]:
# Generate index intance in CPU
index_cpu = faiss.IndexFlatL2(EMBEDDING_DIM)
# Move index to GPU
res = faiss.StandardGpuResources()
index = faiss.index_cpu_to_gpu(res, 0, index_cpu)
# Add all embeddings
index.add(embeddings)
print(f"Total vectors indexed: {index.ntotal}")

Total vectors indexed: 163753


In [ ]:
batch_size = 10_000
results = []

for i in tqdm(range(0, len(embeddings), batch_size)):
    batch = embeddings[i : i + batch_size]
    distances, indices = index.search(batch, k=K_NEIGHBORS)
    results.append(indices[:, 1:])

all_neighbors = np.concatenate(results, axis=0)
print(f"\nSearch complete. Total result sets: {len(results)}")

100%|██████████| 17/17 [00:15<00:00,  1.12it/s]


Search complete. Total result sets: 17


> *Note: KNN took 3 seconds to find 100 neighbors for all 163,753 articles.*

### Approximate Nearest Neighbors (ANN)

In [ ]:
# Create quantizer for ANN search
quantizer = faiss.IndexFlatL2(EMBEDDING_DIM)
# Create ANN CPU instance
n_list = 512
index_cpu = faiss.IndexIVFFlat(quantizer, EMBEDDING_DIM, n_list, faiss.METRIC_L2)
# Move ANN to GPU
res = faiss.StandardGpuResources()
index = faiss.index_cpu_to_gpu(res, 0, index_cpu)
# Train index with all our embeddings
index.train(embeddings)
# Add embeddings to search
index.add(embeddings)
print(f"Total vectors indexed: {index.ntotal}")

Total vectors indexed: 163753


In [ ]:
batch_size = 10_000
index.nprobe = 64

ann_results = []
for i in tqdm(range(0, len(embeddings), batch_size)):
    batch = embeddings[i : i + batch_size]
    distances, indices = index.search(batch, k=K_NEIGHBORS)

    for row_i, idx_row in enumerate(indices):
        global_i = i + row_i
        neighbors = idx_row[idx_row != global_i][:K_NEIGHBORS - 1]
        ann_results.append(neighbors)

all_neighbors_ann = np.array(ann_results)
print(f"\nSearch complete. Total result sets: {len(all_neighbors_ann)}")

> *Note: ANN took 16 seconds to find 100 neighbors for all 163,753 articles.*

#### Evaluation

In this subsection, we compare the results of exact k-Nearest Neighbors (KNN) with those of Approximate Nearest Neighbors (ANN).  

To evaluate retrieval quality, we compute the intersection between the neighbor sets returned by both methods over 50,000 query examples. This allows us to measure how closely the ANN results approximate the true nearest neighbors obtained via exact KNN.

The overlap between both neighbor sets serves as a proxy for recall and helps quantify the trade-off between speed and retrieval accuracy.

In [ ]:
eval_size = 50_000
found = 0

for i in range(eval_size):
    # Count how many of the true KNN neighbors were captured by ANN
    intersection = np.intersect1d(all_neighbors[i], all_neighbors_ann[i])
    found += len(intersection)

recall = found / (eval_size * (K_NEIGHBORS - 1))

print(f"Evaluation for {eval_size} samples:")
print(f"Final ANN Recall@k: {recall:.4f}")

Evaluation for 50000 samples:
Final ANN Recall@k: 0.9654


### Results

The results show that **exact KNN** required **only 3 seconds** to retrieve the top-*k* most similar articles for all query examples, whereas **ANN** completed the same task in **16 seconds**, representing approximately **4× longer runtime**. Despite this difference, both methods still operate within very fast computation times, likely due to the availability of **high-performance GPUs**.

In terms of retrieval quality, the **Recall@k** metric, which measures the proportion of true nearest neighbors (as determined by exact KNN) that are successfully retrieved by ANN — is **0.9654**.


<br/>

#### Algorithm Conclusion

Given that **exact KNN is both faster and retrieves the true nearest neighbors**, we select **KNN** as the similarity search method for the remainder of the research pipeline.

If the dataset size increases significantly in the future, **ANN may become the preferable option**, as approximate methods typically scale better for very large datasets.

#### Articles Pair Generation

In [40]:
original_ids = df["id"].values

article_ids = original_ids.repeat(K_NEIGHBORS - 1)
neighbor_ids = original_ids[all_neighbors.flatten()]

pairs_df = pd.DataFrame({
    "article_id": article_ids,
    "neighbor_id": neighbor_ids
})
print(f"Generated {len(pairs_df)} candidate pairs with original IDs")

Generated 16211547 candidate pairs with original IDs


> *Note: For each article, we generate 100 candidate pairs using the following structure:*  
> *(A, A₁), (A, A₂), ..., (A, Aₖ),*  
> *where each Aᵢ represents one of the top-*k* most similar articles retrieved during the similarity search stage.*

> *Total pairs: 16,211,547.*

#### Populate Pairs Dataframe

In [ ]:
cols_to_join = ["id", "title", "content", "event_id", "entities", "entities_ids"]

article_meta = df_entities[cols_to_join].add_prefix("article_").rename(columns={"article_id": "article_id"})
neighbor_meta = df_entities[cols_to_join].add_prefix("neighbor_").rename(columns={"neighbor_id": "neighbor_id"})

pairs_df = (
    pairs_df
    .merge(article_meta, on="article_id", how="left")
    .merge(neighbor_meta, on="neighbor_id", how="left")
)
print("Article pairs populated successfully with title, content and event_id")

Article pairs populated successfully with title, content and event_id


---

## 3. Pair Comparison

### 3.1 Cross-Encoder Classifier

In this section, we use our **custom fine-tuned cross-encoder classifier** available through the Hugging Face Hub: `Juanillaberia/articles-pairs-event-detection`. The model is based on **ModernBERT** and was fine-tuned on approximately **20k pairs of news article headlines** to predict whether two articles refer to the **same real-world event**.

Additionally, if the **publication date of each article** is provided along with the headlines, the model can leverage a **post-hoc temporal feature** based on the **difference between article dates**. This approach provides an **approximate 3% improvement in evaluation metrics**, as observed during experimentation.

The workflow for this step is as follows:

1. **Load the model and tokenizer** from the Hugging Face Hub.
2. **Run inference on the generated article pairs**.

The predicted **probability** that two articles belong to the same event will be used as the primary similarity score in the next stage of the pipeline to determine whether articles should be grouped together.

#### Load Model & Tokenizer


In [ ]:
MODEL_NAME = "Juanillaberia/articles-pairs-event-detection"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

#### Tokenize & Predict Article Pairs

First, we create a **custom tokenization function** that allows article pairs to be processed in **batches**. This approach improves inference efficiency and enables seamless integration with the Hugging Face tokenizer.

We leverage the tokenizer’s `text` and `text_pair` parameters, which allow us to pass **two headlines as separate inputs**. The tokenizer automatically formats them as a pair by inserting the appropriate **`[SEP]` (separator) token**, enabling the model to treat the two headlines as a single sequence containing two segments.

After tokenizing all article pairs, we perform **batch inference** using the fine-tuned model to obtain the **probability that each pair of articles refers to the same real-world event**.

In [ ]:
def tokenize_examples(pairs_df: pd.DataFrame):
    """
    Handles tokenization for all pairs.

    Args:
        pairs_df (pd.DataFrame): DataFrame containing the data from both articles (a pair of articles).

    Returns:
        Tokenized pairs ready to be used by HuggingFace model.
    """
    text_a = pairs_df["article_title"].astype(str)
    text_b = pairs_df["neighbor_title"].astype(str)

    return tokenizer(
        text=text_a.tolist(),
        text_pair=text_b.tolist(),
        truncation=True,
        return_tensors="pt",
        max_length=128,
        padding=True
    )

In [34]:
# Setup inference variables
OUTPUT_PATH = "/content/drive/MyDrive/Research/articles_grouping/datasets/results/intermediate/pairs_scored_2500_new.csv"
BATCH_SIZE = 1024

In [ ]:
# Prepare model for inference based on device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()
model.to(device)

In [ ]:
all_probs = []

torch.set_float32_matmul_precision("high")

if len(pairs_df) > 0:
    print(f"Running inference on {len(pairs_df):,} pairs using {device}...")
    print(f"Saving incremental results to: {OUTPUT_PATH}")

    # Clear file if exists to start fresh
    if os.path.exists(OUTPUT_PATH):
        print("Warning: Overwriting existing output file.")
        os.remove(OUTPUT_PATH)

    for i in tqdm(range(0, len(pairs_df), BATCH_SIZE)):
        # 1. Slice the batch
        batch_df = pairs_df.iloc[i : i + BATCH_SIZE]

        # 2. Tokenize
        inputs = tokenize_examples(batch_df)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # 3. Inference
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1)[:, 1]
            batch_probs = probs.cpu().numpy()

        # 4. In-memory storage
        all_probs.append(batch_probs)

        # 5. Disk storage
        cols_to_save = ["article_id", "neighbor_id", "article_event_id", "neighbor_event_id"]
        batch_results = batch_df[cols_to_save].copy()
        batch_results["similarity_score"] = batch_probs
        #batch_results["prediction"] = (batch_probs > 0.25).astype(int)

        batch_results.to_csv(OUTPUT_PATH, mode='a', index=False, header=(i==0))

    # 6. Finalize in-memory dataframe
    pairs_df["similarity_score"] = np.concatenate(all_probs)
    #pairs_df["prediction"] = (pairs_df["similarity_score"] > 0.25).astype(int)
    print("\nInference complete. Scores added to pairs_df and saved to Drive.")
else:
    print("No pairs to process after filtering.")

> *Note: The cross-encoder classifier took 1 hour, 30 minutes and 10 seconds to classify 16,211,547 examples running on a NVIDIA A100.*

#### Load Cross-Encoder Results

The inference batching system that as implemented above saved every batch to a local .csv, allowing us to load the results when needed and not have to re-calculate them on each iteration of the research.

In [35]:
cross_encoder_results = pd.read_csv(filepath_or_buffer=OUTPUT_PATH)
print(f"Loaded results: {len(cross_encoder_results)} pairs")

Loaded results: 16211547 pairs


> *Note: This section will be revisited during the **error analysis stage**, where we will analyze how the results change after incorporating the subsequent components of the pipeline.*

### 3.2 Entity Extraction & Analysis

We compare the entities sets of each pair using **Jaccard similarity**, which measures the proportion of shared entities between two sets by dividing the size of their intersection by the size of their union.


#### Entities Comparison & Analysis


In [ ]:
def parse_entity_string(s):
    if isinstance(s, str):
        return re.findall(r"'([^']+)'", s)
    return list(s)

def jaccard_vectorized(series_a, series_b):
    results = []
    for a, b in zip(series_a, series_b):
        set_a = set(parse_entity_string(a))
        set_b = set(parse_entity_string(b))
        union = set_a | set_b
        results.append(len(set_a & set_b) / len(union) if union else 1.0)
    return results

pairs_df["jaccard_score"] = jaccard_vectorized(
    pairs_df["article_entities_ids"],
    pairs_df["neighbor_entities_ids"]
)

### 3.3 Final Score Calculation

The final similarity score for each article pair is computed by combining the **cross-encoder score** and the **Jaccard similarity score** obtained from the entity comparison. The two components are aggregated using a **weighted sum**, allowing each signal to contribute differently to the final decision.

In this setup, the **cross-encoder score receives a weight of 0.85**, while the **Jaccard entity similarity receives a weight of 0.15**.

        S = 0.85 * S_cross + 0.15 * S_jaccard

> *Note: The initial weighting assigned 0.7 to the cross-encoder score and 0.3 to the Jaccard entity similarity score. Error analysis showed that true same-event pairs often had low Jaccard scores (mean 0.28) compared to the cross-encoder scores (mean 0.56), indicating that entity overlap was penalizing valid pairs. Because articles about the same event may emphasize different aspects and share few entities, the Jaccard weight was reduced to 0.15 and the cross-encoder weight increased to 0.85, improving the retention of true event pairs.*

In [ ]:
CROSS_ENCODER_WEIGHT = 0.85
JACCARD_WEIGHT = 0.15

pairs_df_analyzed = pairs_df.copy()

if "similarity_score" not in pairs_df_analyzed.columns:
    pairs_df_analyzed = pairs_df_analyzed.merge(
        cross_encoder_results[["article_id", "neighbor_id", "similarity_score"]],
        on=["article_id", "neighbor_id"],
        how="left"
    )

if "jaccard_score" not in pairs_df_analyzed.columns:
    pairs_df_analyzed["jaccard_score"] = pairs_df_analyzed.apply(
        lambda row: jaccard_vectorized(row["article_entities_ids"], row["neighbor_entities_ids"]),
        axis=1
    )

pairs_df_analyzed["final_score"] = (
    (pairs_df_analyzed["similarity_score"] * CROSS_ENCODER_WEIGHT) +
    (pairs_df_analyzed["jaccard_score"] * JACCARD_WEIGHT)
)

print("Final scores calculated successfully.")

Final scores calculated successfully.


#### Build Final Dataframe

In [ ]:
final_pairs_df = pairs_df_analyzed[[
    "article_id",
    "neighbor_id",
    "final_score",
    "article_event_id",
    "neighbor_event_id",
    "jaccard_score",
    "similarity_score"
]].copy()

print(f"Final dataframe created with {len(final_pairs_df):,} entries.")

Final dataframe created with 16,211,547 entries.


In [ ]:
FINAL_RESULTS_PATH = "/content/drive/MyDrive/Research/articles_grouping/datasets/results/final/final_scores_new.csv"

In [ ]:
# Save results dataframe
final_pairs_df.to_csv(FINAL_RESULTS_PATH, index=False)

---
## 4. Graph Generation

#### Load Final Results

In [ ]:
final_pairs_df = pd.read_csv(filepath_or_buffer=FINAL_RESULTS_PATH)
print(f"Loaded results: {len(final_pairs_df)}")

Loaded results: 16211547


### 4.1 Filter Pairs

In this section we will filter the generated pairs based on their `final_score`, by doing this we don't need to perform this overhead in the graph generation as we know that all rows here are connections between nodes.

Not all nodes will be connected as only articles from a same cluster will be conected together.

In [ ]:
SIMILARITY_THRESHOLD = 0.5

final_pairs_df = final_pairs_df[final_pairs_df["final_score"] >= SIMILARITY_THRESHOLD]
print(f"Remaining pairs after filtering: {len(final_pairs_df)}")

Remaining pairs after filtering: 4476375


In [ ]:
coverage = pd.concat([final_pairs_df["article_id"], final_pairs_df["neighbor_id"]]).nunique()
print(f"Article coverage: {coverage}/{len(df)} - {(coverage/len(df))*100:.1f}%")

Article coverage: 146402/163753 - 89.4%


### 4.2 Graph Construction

In this step, we construct a **graph representation of the articles** using the filtered pairs obtained in the previous stages. Each **node** in the graph represents an article, and an **edge** is created between two nodes when the corresponding article pair appears in the filtered dataset.

The **edge weight** corresponds to the final similarity score computed in the previous step.

Once the graph is constructed, we apply the **Leiden community detection algorithm** to identify clusters of articles that likely refer to the **same real-world event**.

In [ ]:
G = cugraph.Graph()
G.from_pandas_edgelist(final_pairs_df,
                       source="article_id",
                       destination="neighbor_id",
                       edge_attr="final_score",
                       renumber=True)

### 4.3 Leiden Resolution Selection

To determine the optimal **resolution parameter** for the Leiden algorithm, we performed a sweep across multiple values using the same graph constructed with a similarity threshold of **0.5**. The results are summarized below:

| Resolution | Communities | Median Size | 75th Percentile | Max Size |
|-------------|-------------|-------------|------------------|----------|
| 1 | 2,875 | 353 | 3,590 | 103,308 |
| 3 | 1,421 | 107 | 4,206 | 4,654 |
| 20 | 4,552 | 56 | 434 | 4,500 |
| **200** | **6,015** | **8** | **40** | **203** |
| 1,000 | 17,473 | 3 | 10 | 101 |

<br/>

The value **resolution = 200** was selected because it provides a **balanced cluster size distribution**. It produces a reasonable number of communities, a maximum cluster size approaching the gold-standard ceiling, and a healthy 75th percentile, without the **over-fragmentation** observed at **resolution = 1,000**.

In [ ]:
parts, modularity = cugraph.leiden(G, max_iter=100, resolution=200, random_state=42, theta=1.0)

In [ ]:
print(f"Unique partitions: {parts['partition'].nunique()}")
print(f"Modularity: {modularity:.4f}")

Unique partitions: 6538
Modularity: 0.9306


### 4.3 Evaluation

In this section, we evaluate the quality of the generated clusters against the **gold standard event labels**. Each article in the dataset is associated with a ground-truth `event_id`, which serves as the reference for all evaluation metrics.

We report the following metrics:

- **Homogeneity**  
  Measures cluster purity. A score of **1.0** means every cluster contains articles from only one event.

- **Completeness**  
  Measures event recovery. A score of **1.0** means all articles belonging to the same event are assigned to the same cluster.

- **V-Measure**  
  The harmonic mean of homogeneity and completeness, providing a single balanced quality score.

- **Average Intra-cluster Cosine Similarity**  
  Measures the semantic coherence of articles within each cluster using their embeddings.

Before computing the metrics, articles assigned a partition value of `-1` are converted to **unique singleton identifiers**. This prevents them from being treated as a single artificial cluster by the evaluation framework.

In [ ]:
evaluation_df = df.copy()
evaluation_df = evaluation_df.drop(labels=["title", "content", "url", "text"], axis=1)

evaluation_df = (
    evaluation_df
    .merge(parts.to_pandas(), left_on="id", right_on="vertex", how="left")
)

In [ ]:
evaluation_df["partition"] = evaluation_df["partition"].fillna(-1).astype(int)
print("NaN values in 'partition' column replaced with -1.")

NaN values in 'partition' column replaced with -1.


Articles that did not appear in any surviving pair after threshold filtering — either because they had no neighbors above the similarity threshold or because they were absent from the KNN candidate pool entirely — were assigned a partition label of `-1`. If left as-is, all `-1` articles would be treated by the evaluation metrics as belonging to a single large artificial cluster, which would incorrectly inflate homogeneity if those articles span many different events. To avoid this distortion, each `-1` article was reassigned a unique negative integer identifier, effectively treating every unassigned article as its own singleton cluster. This ensures the evaluation reflects the true behavior of the pipeline, unassigned articles are isolated, not grouped together.

In [ ]:
# Assign unique IDs to singletons
partition_fixed = evaluation_df["partition"].copy()
noise_mask = partition_fixed == -1
partition_fixed[noise_mask] = range(-1, -noise_mask.sum() - 1, -1)

In [ ]:
y_true = evaluation_df["event_id"]
y_pred = partition_fixed

homogeneity, completeness, v_measure = homogeneity_completeness_v_measure(y_true, y_pred)
print(f"Homogeneity Score: {homogeneity:.4f}")
print(f"Completeness Score: {completeness:.4f}")
print(f"V-Measure Score: {v_measure:.4f}")

Homogeneity Score: 0.9474
Completeness Score: 0.8589
V-Measure Score: 0.9010


In [ ]:
def calculate_intra_sim(df, embeddings, n_clusters_to_sample=4_000):
    valid_clusters = df[df["partition"] != -1]["partition"].unique()
    sampled_clusters = np.random.choice(valid_clusters, min(len(valid_clusters), n_clusters_to_sample), replace=False)

    similarities = []
    for cid in sampled_clusters:
        indices = df[df["partition"] == cid].index
        if len(indices) > 1:
            cluster_embeds = embeddings[indices]
            sim_matrix = cosine_similarity(cluster_embeds)
            # Take upper triangle to avoid diagonal and double counting
            upper_sims = sim_matrix[np.triu_indices(len(indices), k=1)]
            similarities.append(np.mean(upper_sims))

    return np.mean(similarities) if similarities else 0

avg_sim = calculate_intra_sim(evaluation_df, embeddings)
print(f"\nAverage Intra-cluster Cosine Similarity (sampled): {avg_sim:.4f}")


Average Intra-cluster Cosine Similarity (sampled): 0.9573


### 4.4 Results

After completing the three processing and data extraction stages, we generated the **final article similarity graph** and applied the **Leiden community detection algorithm** to identify event clusters.

Before constructing the graph, we applied an additional **similarity filtering step with a threshold of 0.5**. This value was selected after performing a **threshold sweep**, where it achieved the best overall clustering metrics.

#### Graph Statistics

- **Article coverage:** 146,402 / 163,753 (**89.4%**)
- **Unique partitions (clusters):** 6,538
- **Modularity:** 0.9306

Articles that were **not included in any pair** (either because they did not pass the KNN retrieval stage or were filtered out in later steps) were labeled as **noise articles** using the label `-1`. These entries were later converted into **unique identifiers** within the DataFrame so they could be treated as **singleton clusters**. This prevents them from forming a large artificial noise cluster that could distort the evaluation metrics.

#### Clustering Quality Metrics

- **Homogeneity:** 0.9474  
- **Completeness:** 0.8589  
- **V-Measure:** 0.9010  

These results indicate that the clustering approach produces **highly coherent event groups**, with most clusters containing articles that belong to the same ground-truth event.

#### Cluster Cohesion

To further evaluate the internal consistency of the clusters, we computed the **Average Intra-cluster Cosine Similarity** (using sampling):

- **Average Intra-cluster Cosine Similarity:** 0.9573

This high similarity value suggests that articles grouped within the same cluster are **strongly related in semantic space**, supporting the effectiveness of the event detection pipeline.



---

# Conclusion

We started with a solid baseline using **HDBSCAN and semantic similarity** to cluster articles into events. Its best results achieved **~98% homogeneity**, meaning that most articles inside a cluster belonged to the same original event. However, **completeness** did not perform as well, reaching **~75%**, which means that a significant portion of articles belonging to an event were either marked as noise or included in a different cluster.

In addition, the **intra-cluster similarity** was **~88%**, indicating that the articles inside a cluster shared strong semantic similarity.

Something very important to highlight is the proportion of articles labeled as **noise** by the baseline model. The HDBSCAN approach left **~38.7% of the articles as noise**, resulting in a **coverage of only 61.3%** of the dataset.

These results prove that using a simple approach can yield good results with some limitations. However, this baseline may struggle in production-level applications that integrate this approach. **HDBSCAN is effective at clustering by topic or semantic similarity**, but when two events are very similar it can merge them, generating larger clusters that actually contain multiple distinct events.

<br/>

From this baseline we designed a **multi-signal approach** to test whether adding additional analysis layers focused on more specific parts of the articles could improve the results.

The pipeline works as follows:

1. **Semantic retrieval using KNN** to generate candidate article pairs.
2. Each pair is evaluated using a **fine-tuned cross-encoder model**, which produces a similarity score.
3. Using the **extracted and normalized entities**, we compute **Jaccard similarity** between the entity sets.
4. The scores are combined to generate a **final similarity score**.
5. A **graph is constructed**, and **Leiden community detection** is applied to generate the final clusters.

Using the configurations and parameters described throughout the notebook, this approach achieved **better results in three of the four evaluated metrics compared to the baseline**:

- **Homogeneity:** 0.9474  
- **Completeness:** 0.8589  
- **V-Measure:** 0.9010  
- **Average Intra-cluster Cosine Similarity:** 0.9573  

Even though **homogeneity decreased by ~4%**, **completeness**, **V-Measure**, and **average intra-cluster similarity** increased by approximately **15%**, **6%**, and **9%** respectively.

One of the most important metrics to consider is the **coverage** of the final clustering. With the proposed pipeline, only **~10% of the total articles were labeled as noise**, representing a **28.7% reduction compared to the HDBSCAN baseline**.

In addition, we believe that this approach provides **greater robustness for grouping real-world events**.

---

# Future Work

Below we outline possible improvements and additional experiments that could further improve the pipeline:

- **Weighted Jaccard similarity**  
  Assign higher importance weights to **PER** and **ORG** entities compared to **LOC** and **MISC**.  
  The **MISC** category may even be removed entirely since it often introduces noise.

- **Hybrid article retrieval**  
  Instead of relying only on **KNN**, we plan to test combining it with **BM25**.  
  This would allow the retrieval stage to capture not only semantic similarity but also lexical similarity (names, dates, quotes) that embeddings might dilute.  
  A **fusion step** would then select the final *K* articles used to generate candidate pairs.

- **Re-train the cross-encoder with more data**  
  The current dataset used for training the cross-encoder contains only **~20k article pairs** and is **imbalanced toward negative examples**.  

  The goal is to construct a new dataset with approximately **200k examples**, extracted from the original clustering dataset. This dataset would include both **positive pairs** and **hard negatives**, while ensuring that **no articles used in this pipeline are included**, preventing data leakage.

  This would allow the cross-encoder to see significantly more examples while also training on a **balanced dataset**, which should improve the model’s confidence and prediction quality.